# Weekly Music Recommendation Pipeline

This notebook demonstrates how Kale can convert a Jupyter notebook into a Kubeflow Pipeline.

**What this pipeline does:**
- Loads user listening history and song catalog
- Trains a collaborative filtering recommendation model
- Generates personalized playlists for all users
- Evaluates playlist quality

**Pipeline DAG:**
```
load_data → build_matrix → train_model → ┬ detect_drift
                                         └ generate_playlists → evaluate → metrics
```

In [ ]:
# Libraries used throughout the pipeline
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from lightfm import LightFM
from lightfm.evaluation import precision_at_k
import json
from datetime import datetime, timedelta
from collections import defaultdict, Counter

In [ ]:
# Helper functions available to all pipeline steps

def calculate_implicit_rating(play_count, completion_rate, skip_count, save_count):
    """
    Convert listening behavior into 0-10 implicit rating.
    High play count + completion + saves = strong positive signal.
    Skips = negative signal.
    """
    base_score = play_count * 0.5
    completion_bonus = completion_rate * 3
    skip_penalty = skip_count * -1.5
    save_bonus = save_count * 2
    
    rating = base_score + completion_bonus + skip_penalty + save_bonus
    return max(0, min(10, rating))


In [ ]:
# Load listening events (user behavior over last 8 weeks)
DATA_URL = "https://raw.githubusercontent.com/ada333/Kale-music-recommendation-use-case/main/data"
listening_data = pd.read_csv(f"{DATA_URL}/listening_events.csv")

# Load song catalog (metadata for all songs)
songs = pd.read_csv(f"{DATA_URL}/songs.csv")

# Calculate implicit ratings from behavior
listening_data['rating'] = listening_data.apply(
    lambda row: calculate_implicit_rating(
        row['play_count'], 
        row['completion_rate'], 
        row['skip_count'], 
        row['save_count']
    ), 
    axis=1
)

# Summary stats
n_users = listening_data['user_id'].nunique()
n_songs_listened = listening_data['song_id'].nunique()

print(f"   Number of users: {n_users}")
print(f"   Number of songs listened to: {n_songs_listened}")

In [ ]:
# Create user and song ID mappings
user_ids = listening_data['user_id'].unique()
song_ids = songs['song_id'].unique()

user_to_idx = {user_id: idx for idx, user_id in enumerate(user_ids)}
song_to_idx = {song_id: idx for idx, song_id in enumerate(song_ids)}


# Build sparse matrix (most cells are zero)
rows = []
cols = []
data = []

for _, row in listening_data.iterrows():
    user_idx = user_to_idx[row['user_id']]
    song_idx = song_to_idx[row['song_id']]
    rating = row['rating']
    
    rows.append(user_idx)
    cols.append(song_idx)
    data.append(rating)

interaction_matrix = csr_matrix(
    (data, (rows, cols)), 
    shape=(len(user_ids), len(song_ids))
)

In [ ]:
# Initialize model
model = LightFM(
    no_components=32,  # 32-dimensional embeddings
    loss='warp',        # WARP loss optimizes for ranking
    random_state=42
)

for _ in range(10):
    model.fit_partial(interaction_matrix, epochs=1)

# Extract learned embeddings
user_embeddings = model.user_embeddings
song_embeddings = model.item_embeddings

# Reverse mapping needed for generating playlists
idx_to_song = {idx: song_id for song_id, idx in song_to_idx.items()}

In [ ]:
# Check genre distribution

current_genre_dist = listening_data.merge(songs, on='song_id')['genre'].value_counts(normalize=True)

# Simulated data for demo - in real life scenario this data would be fetched from somewhere
historical_baseline = {
    'Rock': 0.22,
    'Electronic': 0.20,
    'Rap': 0.18,
    'Czech songs': 0.12
}

max_shift = 0
for genre in historical_baseline:
    current = current_genre_dist.get(genre, 0)
    historical = historical_baseline[genre]
    shift = abs(current - historical) * 100
    max_shift = max(max_shift, shift)

print(f"\n  Maximum genre shift: {max_shift:.1f} percentage points")

drift_status = 'NORMAL'
if max_shift > 15:
    drift_status = 'MAJOR_DRIFT'
elif max_shift > 8:
    drift_status = 'MODERATE_DRIFT'

drift_report = {
    'max_genre_shift': max_shift,
    'status': drift_status,
    'timestamp': datetime.now().isoformat()
}

In [ ]:
# Generates playlists for all users (for demo only 10)
NUM_DEMO_USERS = 10
PLAYLIST_SIZE = 30
NUM_CANDIDATES = 100 
all_playlists = {}


for user_idx, user_id in enumerate(user_ids[:NUM_DEMO_USERS]):
    user_emb = user_embeddings[user_idx].reshape(1, -1)
    
    similarities = cosine_similarity(user_emb, song_embeddings)[0]
    user_history_songs = listening_data[listening_data['user_id'] == user_id]['song_id'].values

    user_listened = songs[songs['song_id'].isin(user_history_songs)]
    language_counts = user_listened['language'].value_counts()
    

    non_instrumental = language_counts[language_counts.index != 'Instrumental']
    user_language = non_instrumental.index[0] if len(non_instrumental) > 0 else 'English'

    candidates = []
    for song_idx, similarity in enumerate(similarities):
        song_id = idx_to_song[song_idx]
        if song_id not in user_history_songs:
            song_info = songs[songs['song_id'] == song_id].iloc[0]
            candidates.append({
                'song_id': song_id,
                'title': song_info['title'],
                'artist': song_info['artist'],
                'genre': song_info['genre'],
                'language': song_info['language'],
                'similarity': similarity
            })

    candidates_df = pd.DataFrame(candidates).nlargest(NUM_CANDIDATES, 'similarity')
    
    # LANGUAGE FILTERING: Keep only user's language + Instrumental
    filtered = candidates_df[
        (candidates_df['language'] == user_language) |
        (candidates_df['language'] == 'Instrumental')
    ]
    
    # If not enough candidates after filtering, use all candidates
    if len(filtered) < PLAYLIST_SIZE:
        filtered = candidates_df
    
    final_playlist = filtered.head(PLAYLIST_SIZE)
    
    # Store playlist
    all_playlists[user_id] = {
        'songs': final_playlist.to_dict('records'),
        'diversity_score': final_playlist['genre'].nunique(),
        'avg_similarity': final_playlist['similarity'].mean(),
        'user_language': user_language
    }
print(f"\n Generated {len(all_playlists)} playlists with {PLAYLIST_SIZE} songs each")

In [ ]:
# Evaluate playlist quality
# Metrics: genre diversity (unique genres per playlist), recommendation similarity,
# and filter bubble rate (playlists dominated by a single genre >70%)

diversity_scores = [p['diversity_score'] for p in all_playlists.values()]
similarity_scores = [p['avg_similarity'] for p in all_playlists.values()]

# Count playlists with >70% songs from a single genre (low diversity / "filter bubble" effect)
low_diversity_count = 0
for playlist in all_playlists.values():
    songs_df = pd.DataFrame(playlist['songs'])
    top_genre_pct = songs_df['genre'].value_counts().iloc[0] / len(songs_df)
    if top_genre_pct > 0.7:
        low_diversity_count += 1

filter_bubble_rate = low_diversity_count / len(all_playlists)

# Average metrics (cast to Python float for JSON serialization)
avg_diversity = float(np.mean(diversity_scores))
avg_similarity = float(np.mean(similarity_scores))
filter_bubble_rate = float(filter_bubble_rate)


all_checks_passed = (
    avg_diversity >= 5.0 and
    filter_bubble_rate < 0.15 and
    avg_similarity >= 0.75
)

print(f"  Quality check: {'PASSED' if all_checks_passed else 'FAILED'}")

In [ ]:
print(avg_diversity)
print(avg_similarity)
print(filter_bubble_rate)